# From 3D CFD to reduced-order haemodynamics

## Geometry transfer, information preservation, and calibration of a tortuous vessel

**Scientific question.** How can a three-dimensional CFD model of a tortuous vessel be transformed into a reduced-order representation, what haemodynamic information is preserved during simplification, and which parameters must be calibrated for a hybrid 1D–0D model to reproduce the CFD reference solution?

This notebook is the **single reproducible source** for the figures, tables, and numerical results in the final report. It follows the scientific workflow rather than the order in which scripts were developed.

### Scope and evidence levels

| Label | Meaning |
|---|---|
| **Geometry-derived** | Computed from the vessel surface or centerline; transferable without CFD |
| **CFD-derived** | Computed from the converged 3D solution or its exported samples |
| **Model-derived** | Predicted by an analytical, 0D, or 1D representation |
| **Calibration input** | Not identifiable from lumen geometry alone |
| **Unavailable** | Not present in the archived exports; no value is invented |

Pressure exported by incompressible OpenFOAM is kinematic pressure, \(p/\rho\) [m²/s²], and is converted to Pa using \(\rho=1060\) kg/m³. The analysis is steady; it does not claim pulsatile validation.


## 0. Reproducible setup and data audit


In [ ]:
from pathlib import Path
import csv, math, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    raise RuntimeError("Run this notebook from the repository root.")
OUT = ROOT / "output" / "final_publication"
OUT.mkdir(parents=True, exist_ok=True)

RHO = 1060.0             # kg/m3
MU = 0.004               # Pa s
MMHG_PA = 133.322368
SLICE_ORDER = ["inlet", "midslice", "outlet"]
COLORS = {"CFD (reference)": "#222222", "Poiseuille": "#D55E00",
          "0D native": "#0072B2", "0D N=50": "#56B4E9", "1D Nb02": "#009E73"}

plt.rcParams.update({
    "figure.dpi": 125, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "figure.constrained_layout.use": True,
})

def read_rows(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def num(row, key):
    return float(row[key])

def savefig(fig, name):
    path = OUT / name
    fig.savefig(path, bbox_inches="tight")
    print("saved:", path)

def show_table(headers, rows, formats=None):
    # Dependency-free notebook table display.
    formats = formats or {}
    def fmt(v, j):
        if v is None: return "—"
        if isinstance(v, (float, np.floating)):
            return formats.get(j, "{:.4g}").format(v)
        return str(v)
    text = ["| " + " | ".join(headers) + " |",
            "| " + " | ".join(["---"] * len(headers)) + " |"]
    text += ["| " + " | ".join(fmt(v, j) for j, v in enumerate(r)) + " |"
             for r in rows]
    try:
        from IPython.display import Markdown, display
        display(Markdown("\n".join(text)))
    except ImportError:
        print("\n".join(text))

inputs = {
    "centerline graph": ROOT / "data/graph_mr_025.vtp",
    "surface mesh": ROOT / "openfoam/segment_test_V2/constant/triSurface/segment_test_closed_ascii.stl",
    "geometry summary": ROOT / "output/segment_test_v2_V1/segment_geometry.csv",
    "curvature summary": ROOT / "output/segment_test_v2_V1/segment_curvature_summary.csv",
    "1D geometry profile": ROOT / "output/reduced_order_models/segment_1D_geometry_profile.csv",
    "pressure slices": ROOT / "output/06_clear_results/pressure_slice_summary.csv",
    "flow slices": ROOT / "output/segment_test_v2_V1/segment_flow_summary.csv",
    "velocity summary": ROOT / "output/06_clear_results/velocity_distribution_summary.csv",
    "model comparison": ROOT / "output/reduced_order_models/model_comparison_summary.csv",
    "FirstBlood template": ROOT / "output/reduced_order_models/firstblood_input_template.csv",
}
missing = [str(p) for p in inputs.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Required archived inputs are missing:\n" + "\n".join(missing))
show_table(["Input", "Path", "Status"],
           [(k, str(p.relative_to(ROOT)), "available") for k, p in inputs.items()])


The audit deliberately fails on missing required inputs. Optional field-level quantities are handled later as explicit “unavailable” results. Existing summary CSVs are reused where they are the archived output of expensive CFD or geometry processing; their values are cross-checked against raw slice exports when those exports are available.


## 1. Geometry characterization


The centerline is the bridge from the three-dimensional lumen to reduced models. Arc length, local radius/area, connectivity, tortuosity, and curvature are geometry-derived. Pressure, velocity, flow rate, resistance, and wall shear are not geometry alone and require either CFD, measurement, or a modelling assumption.


In [ ]:
geom_raw = read_rows(inputs["geometry summary"])
geom = {r["quantity"]: float(r["value"]) for r in geom_raw}
curv_raw = read_rows(inputs["curvature summary"])
curv = {r["quantity"]: float(r["value"]) for r in curv_raw}
profile = read_rows(inputs["1D geometry profile"])
s_mm = np.array([num(r, "s_m") * 1e3 for r in profile])
r_mm = np.array([num(r, "r_m") * 1e3 for r in profile])
area_mm2 = np.array([num(r, "area_m2") * 1e6 for r in profile])
kappa_mm = np.array([num(r, "kappa_per_m") / 1e3 for r in profile])

geometry_table = [
    ("Main-path arc length", geom["main path arc length"], "mm", "centerline", "Geometry-derived"),
    ("Inlet–outlet distance", geom["inlet-outlet straight distance"], "mm", "centerline endpoints", "Geometry-derived"),
    ("Tortuosity L/D", geom["tortuosity  T = arc / straight"], "—", "centerline", "Geometry-derived"),
    ("Mean radius", np.mean(r_mm), "mm", "centerline radius", "Geometry-derived"),
    ("Minimum radius", np.min(r_mm), "mm", "centerline radius", "Geometry-derived"),
    ("Maximum radius", np.max(r_mm), "mm", "centerline radius", "Geometry-derived"),
    ("Mean area", np.mean(area_mm2), "mm²", "πr²", "Geometry-derived"),
    ("Mean curvature", curv["mean curvature κ_mean"], "mm⁻¹", "centerline derivatives", "Geometry-derived"),
    ("Maximum curvature", curv["max  curvature κ_max"], "mm⁻¹", "centerline derivatives", "Geometry-derived"),
    ("Terminal nodes", geom["number of terminal nodes"], "—", "graph degree", "Geometry-derived"),
]
show_table(["Quantity", "Value", "Unit", "Source", "Evidence"], geometry_table)


In [ ]:
# Publication geometry figure. The archived graph uses separately base64-encoded
# compressed blocks (a legacy VTK writer layout), so decode the required arrays
# explicitly; VTK reads the STL surface directly.
try:
    import vtk, re, base64, struct, zlib
    from vtk.util.numpy_support import vtk_to_numpy

    vtp_bytes = inputs["centerline graph"].read_bytes()
    appended = re.search(rb'<AppendedData[^>]*>\s*_(.*?)\s*</AppendedData>',
                         vtp_bytes, re.S).group(1).strip()
    def vtp_block(offset, next_offset, dtype):
        block = appended[offset:next_offset]
        header = base64.b64decode(block[:24])
        nblocks, block_size, last_size, compressed_size = struct.unpack("<4I", header)
        if nblocks != 1:
            raise ValueError("This compact decoder expects one compressed block per array.")
        raw = zlib.decompress(base64.b64decode(block[24:]))
        return np.frombuffer(raw, dtype=dtype)

    # Offsets are read from the XML declarations, not hard-coded.
    xml_head = vtp_bytes.split(b"<AppendedData", 1)[0]
    offsets = [int(x) for x in re.findall(rb'offset="(\d+)"', xml_head)]
    all_offsets = offsets + [len(appended)]
    xyz = vtp_block(offsets[5], all_offsets[6], "<f4").reshape(-1, 3)
    connectivity = vtp_block(offsets[8], all_offsets[9], "<i8")
    line_offsets = vtp_block(offsets[9], all_offsets[10], "<i8")
    starts = np.r_[0, line_offsets[:-1]]
    lines = [connectivity[a:b] for a, b in zip(starts, line_offsets)]

    sr = vtk.vtkSTLReader(); sr.SetFileName(str(inputs["surface mesh"])); sr.Update()
    surf = sr.GetOutput()
    surf_xyz = vtk_to_numpy(surf.GetPoints().GetData())  # archived STL coordinates are mm
    polys = vtk_to_numpy(surf.GetPolys().GetData()).reshape(-1, 4)[:, 1:]
    lo, hi = surf_xyz.min(axis=0)-1.0, surf_xyz.max(axis=0)+1.0
    in_box = np.all((xyz >= lo) & (xyz <= hi), axis=1)
    visible_lines = [line for line in lines if np.all(in_box[line])]

    fig = plt.figure(figsize=(10.5, 4.6))
    ax = fig.add_subplot(121, projection="3d")
    ax.plot_trisurf(surf_xyz[:,0], surf_xyz[:,1], surf_xyz[:,2], triangles=polys,
                    color="#B9D8E8", alpha=.36, linewidth=0)
    ax.set_title("CFD lumen surface")
    ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]"); ax.set_zlabel("z [mm]")
    ax.set_box_aspect(np.ptp(surf_xyz, axis=0))

    ax = fig.add_subplot(122, projection="3d")
    ax.plot_trisurf(surf_xyz[:,0], surf_xyz[:,1], surf_xyz[:,2], triangles=polys,
                    color="#D9EAF2", alpha=.16, linewidth=0)
    for line in visible_lines:
        q = xyz[line]
        ax.plot(q[:,0], q[:,1], q[:,2], color="#D55E00", lw=1.3)
    ax.scatter(xyz[in_box,0], xyz[in_box,1], xyz[in_box,2], c="#D55E00", s=2)
    ax.set_title("Centerline graph overlaid on lumen")
    ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]"); ax.set_zlabel("z [mm]")
    ax.set_box_aspect(np.ptp(surf_xyz, axis=0))
    savefig(fig, "01_geometry_mesh_centerline.png")
    plt.show()
except Exception as exc:
    warnings.warn("3D geometry rendering unavailable: " + str(exc))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.2, 6.2), sharex=True)
axes[0].plot(s_mm, r_mm, color="#0072B2", lw=1.8)
axes[0].fill_between(s_mm, 0, r_mm, color="#56B4E9", alpha=.18)
axes[0].set_ylabel("Radius [mm]"); axes[0].set_title("Geometry transferred along the main centerline")
axes[1].plot(s_mm, kappa_mm, color="#D55E00", lw=1.5)
axes[1].set_xlabel("Arc length s [mm]"); axes[1].set_ylabel("Curvature κ [mm⁻¹]")
savefig(fig, "02_radius_curvature_profiles.png"); plt.show()


### Geometry interpretation

The main path is 35.893 mm long but its endpoints are only 25.226 mm apart, giving tortuosity \(L/D=1.423\). This is not a straight-pipe geometry. The extracted quantities affect haemodynamics in distinct ways:

- **Length** sets the distance over which viscous pressure loss accumulates; for otherwise fixed conditions, a longer vessel has greater hydraulic resistance.
- **Local radius and area** control both bulk velocity and resistance. Their influence is especially strong because laminar viscous resistance scales approximately as \(r^{-4}\). Consequently, narrow regions contribute disproportionately to the total pressure drop, and replacing the measured radius profile with one mean radius removes important information.
- **Curvature** changes the direction of the velocity field and can promote secondary motion and additional local pressure losses. Curvature is retained as a geometric descriptor, although a classical resistive 0D element does not resolve the associated three-dimensional flow.
- **Tortuosity** summarizes the accumulated departure from a straight vessel. The measured value therefore signals that a straight-tube approximation omits systematic geometric effects rather than only small surface detail.
- **Connectivity** determines available flow paths and becomes essential when the segment is embedded in a vascular network, where pressure and flow must be distributed between branches.

The graph contains three terminal nodes, defining one inlet and two outlets topologically; the CFD comparison below concerns the archived single analyzed path and its named inlet/midslice/outlet planes. “Inlet” and “outlet” are boundary roles, not quantities inferred from pressure.


## 2. CFD reference haemodynamics


The CFD solution is the reference target. The archived surface integrations define bulk pressure and flow, while point exports retain cross-sectional velocity distributions. Full volume streamlines and a wall-resolved WSS field were not exported and are therefore not reconstructed from insufficient data.


In [ ]:
pressure_rows = read_rows(inputs["pressure slices"])
flow_rows = read_rows(inputs["flow slices"])
velocity_rows = read_rows(inputs["velocity summary"])
p_pa = {r["Slice"]: num(r, "Average pressure [Pa]") for r in pressure_rows}
q = {r["location"]: num(r, "Q [m^3/s]") for r in flow_rows}
area = {r["location"]: num(r, "Area [m^2]") for r in flow_rows}
u_bulk = {k: q[k] / area[k] for k in SLICE_ORDER}

p_in, p_mid, p_out = (p_pa[k] for k in SLICE_ORDER)
dp = p_in - p_out
q_ref = q["outlet"]  # same reference used to build the archived ROM comparison
R_cfd = dp / q_ref
grad = dp / (geom["main path arc length"] * 1e-3)
mass_imbalance = (max(q.values()) - min(q.values())) / np.mean(list(q.values())) * 100
re = {k: RHO * u_bulk[k] * (2*np.sqrt(area[k]/np.pi)) / MU for k in SLICE_ORDER}

pressure_summary = [
    ("Inlet pressure", p_in, p_in/MMHG_PA),
    ("Middle pressure", p_mid, p_mid/MMHG_PA),
    ("Outlet pressure", p_out, p_out/MMHG_PA),
    ("Pressure drop", dp, dp/MMHG_PA),
]
show_table(["Pressure quantity", "Pa", "mmHg"], pressure_summary)
show_table(["Derived CFD target", "Value", "Unit"],
           [("Mean axial pressure gradient", grad, "Pa/m"),
            ("Hydraulic resistance Δp/Q", R_cfd, "Pa·s/m³"),
            ("Reference flow rate", q_ref*1e6, "mL/s"),
            ("Max slice-to-slice flow imbalance", mass_imbalance, "%")])


In [ ]:
slice_x = np.array([0, geom["main path arc length"]/2, geom["main path arc length"]])
slice_p = np.array([p_in, p_mid, p_out]) / MMHG_PA
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(SLICE_ORDER, slice_p, color=["#4477AA", "#66CCEE", "#228833"])
axes[0].set_ylabel("Area-averaged pressure [mmHg]"); axes[0].set_title("CFD pressure magnitude")
axes[1].plot(slice_x, slice_p, "o-", color="#222222", lw=1.8)
axes[1].set_xlabel("Centerline position [mm]"); axes[1].set_ylabel("Pressure [mmHg]")
axes[1].set_title("Pressure along the analyzed vessel")
savefig(fig, "03_cfd_pressure.png"); plt.show()


In [ ]:
# Load raw slice velocity samples and cross-check archived statistics.
raw_velocity = {}
point_files = {
    "inlet": ROOT/"data_parafoam/inlet_point_u.csv",
    "midslice": ROOT/"data_parafoam/midsclice_point_u.csv",
    "outlet": ROOT/"data_parafoam/outlet_point_u.csv",
}
for name, path in point_files.items():
    rows = read_rows(path)
    uvw = np.array([[num(r, "U:0"), num(r, "U:1"), num(r, "U:2")] for r in rows])
    xyzp = np.array([[num(r, "Points:0"), num(r, "Points:1"), num(r, "Points:2")] for r in rows])
    raw_velocity[name] = {"mag": np.linalg.norm(uvw, axis=1), "xyz": xyzp}

velocity_table = []
for name in SLICE_ORDER:
    mag = raw_velocity[name]["mag"]
    velocity_table.append((name, len(mag), np.mean(mag), np.max(mag), np.std(mag),
                           u_bulk[name], np.std(mag)/np.mean(mag), re[name]))
show_table(["Slice", "n", "Mean |U|", "Max |U|", "SD |U|", "Q/A", "CV(|U|)", "Re"],
           velocity_table)


In [ ]:
all_u = np.concatenate([raw_velocity[k]["mag"] for k in SLICE_ORDER])
bins = np.linspace(0, np.percentile(all_u, 99.5), 35)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
for k, c in zip(SLICE_ORDER, ["#4477AA", "#EE6677", "#228833"]):
    axes[0].hist(raw_velocity[k]["mag"], bins=bins, density=True, histtype="step", lw=1.8, label=k, color=c)
axes[0].set_xlabel("Velocity magnitude [m/s]"); axes[0].set_ylabel("Probability density")
axes[0].set_title("Cross-sectional velocity distributions"); axes[0].legend(frameon=False)

x = np.arange(3)
axes[1].bar(x-.18, [q[k]*1e6 for k in SLICE_ORDER], width=.36, label="Q [mL/s]", color="#56B4E9")
ax2 = axes[1].twinx()
ax2.plot(x, [u_bulk[k] for k in SLICE_ORDER], "o-", color="#D55E00", label="Q/A")
axes[1].set_xticks(x, SLICE_ORDER); axes[1].set_ylabel("Flow rate [mL/s]")
ax2.set_ylabel("Bulk velocity Q/A [m/s]"); axes[1].set_title("Flow conservation and bulk velocity")
axes[1].grid(False); ax2.grid(False)
savefig(fig, "04_cfd_velocity_flow.png"); plt.show()


In [ ]:
# Project each irregular point cloud onto its two dominant in-plane axes.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
vmin, vmax = 0, np.percentile(all_u, 99)
for ax, name in zip(axes, SLICE_ORDER):
    pts = raw_velocity[name]["xyz"]
    centered = pts - pts.mean(axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    uv = centered @ vh[:2].T * 1e3
    sc = ax.scatter(uv[:,0], uv[:,1], c=raw_velocity[name]["mag"], s=12,
                    cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_aspect("equal"); ax.set_title(name.capitalize())
    ax.set_xlabel("in-plane coordinate [mm]")
axes[0].set_ylabel("in-plane coordinate [mm]")
fig.colorbar(sc, ax=axes, label="|U| [m/s]", shrink=.82)
savefig(fig, "05_velocity_profiles.png"); plt.show()


### What the full CFD simulation preserves

The archived CFD outputs provide area-averaged pressure, pressure drop, flow, bulk velocity, Reynolds number, and non-uniform cross-sectional velocity distributions. Flow imbalance across the three sampled planes is the conservation diagnostic. The differences between mean \(|U|\) and \(Q/A\) are expected: the former is an unweighted sample statistic, whereas the latter is the signed, area-integrated through-plane velocity.

The sample maps reveal profile distortion that a single 0D flow variable cannot encode. **Streamlines, vortices, recirculation, and secondary-flow helicity require the volume vector field and are unavailable in these exports.** A true wall shear stress distribution also requires wall gradients. For context only, the archived Hagen–Poiseuille estimate is reported below; it is not relabelled as CFD WSS.


In [ ]:
wss = read_rows(ROOT/"output/segment_test_v2_V1/segment_wss_summary.csv")
show_table(["Quantity", "Value", "Unit", "Status"],
           [(r["quantity"], float(r["value"]), r["units"],
             "Analytical estimate; not wall-resolved CFD") for r in wss])


## 3. Reduced-order representation


### 3.1 Geometry transfer

The 3D lumen is collapsed to an ordered centerline. Each reduced element receives a length and radius (hence area); graph endpoints and branches define connectivity. Tortuosity and curvature can be retained as descriptors or correction inputs, but a classical Poiseuille 0D element does not use them directly.


In [ ]:
transfer = [
    ("Length", "Centerline arc length", "Poiseuille, 0D, 1D", "Direct"),
    ("Radius", "Centerline radius array", "Poiseuille, 0D, 1D", "Direct; estimator matters"),
    ("Area", "πr²", "0D, 1D", "Direct"),
    ("Connectivity", "Centerline graph", "Network 0D/1D", "Direct"),
    ("Tortuosity", "Arc/chord ratio", "Correction/metadata", "Preserved descriptor"),
    ("Curvature", "Centerline derivatives", "1D correction/metadata", "Preserved descriptor"),
    ("Wall mechanics", "Not in lumen geometry", "Compliant 1D", "Calibration input"),
    ("Outlet impedance", "Not in local geometry", "0D boundary", "Calibration input"),
]
show_table(["Quantity", "Source", "Used by reduced model", "Transfer status"], transfer)


### 3.2 Model definitions

- **Analytical Poiseuille:** one rigid circular tube using total arc length and a representative radius; \(R=8\mu L/(\pi r^4)\).
- **Native 0D:** 123 centerline elements in series, each using its local radius and length.
- **Segmented 0D (50):** the same geometry coarsened to 50 resistive elements.
- **1D Nb02:** variable-area centerline representation from the earlier 1D-style calculation. It is a steady resistive result, not an executed compliant FirstBlood simulation.
- **FirstBlood input:** automatically populated geometry and resistance columns; wall and terminal parameters remain blank by design.


In [ ]:
rom = read_rows(inputs["model comparison"])
show_table(["Model", "R [Pa·s/m³]", "Δp [Pa]", "R error [%]", "Segments"],
           [(r["model"], num(r, "R_Pa_s_m3"), num(r, "dP_Pa"),
             num(r, "R_err_pct"), r["n_segments"]) for r in rom])

fb = read_rows(inputs["FirstBlood template"])
fb_headers = list(fb[0].keys())
automatic = ["segment_id", "length_m", "diameter_m", "area_m2",
             "curvature_mean_per_m", "resistance_segment_raw",
             "resistance_segment_kR", "recommended_division_pts"]
missing_fb = ["wall_stiffness_E_Pa", "wall_thickness_m", "pwv_m_per_s",
              "outlet_resistance_Pa_s_m3", "outlet_compliance_m3_per_Pa"]
show_table(["FirstBlood parameter group", "Columns", "Status"],
           [("Geometry / discretisation", ", ".join(automatic), f"Automatically populated for {len(fb)} elements"),
            ("Wall / outlet physiology", ", ".join(missing_fb), "Missing; requires calibration or literature priors")])
print("Reconstructed FirstBlood input:", inputs["FirstBlood template"].relative_to(ROOT))


## 4. CFD versus reduced-order models


All valid steady bulk quantities are compared together. Velocity profiles, secondary flow, WSS distributions, and vortices are intentionally excluded because the reduced models do not represent them directly. All model pressure drops use the same CFD reference flow, so resistance and pressure-drop percentage errors coincide.


In [ ]:
models = [r["model"] for r in rom]
Rvals = np.array([num(r, "R_Pa_s_m3") for r in rom])
dPvals = np.array([num(r, "dP_Pa") for r in rom])
errs = np.array([num(r, "R_err_pct") for r in rom])
colors = [COLORS.get(m, "#999999") for m in models]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.1))
axes[0].bar(models, Rvals/1e6, color=colors)
axes[0].set_ylabel("Resistance [10⁶ Pa·s/m³]"); axes[0].set_title("Global hydraulic resistance")
axes[1].bar(models, dPvals/MMHG_PA, color=colors)
axes[1].set_ylabel("Pressure drop [mmHg]"); axes[1].set_title(f"Pressure drop at Q = {q_ref*1e6:.3f} mL/s")
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
savefig(fig, "06_model_global_comparison.png"); plt.show()


In [ ]:
# Reconstruct pressure profiles from each model's cumulative resistance.
seg50 = read_rows(ROOT/"output/reduced_order_models/segment_0D_50_segments.csv")
s50 = np.array([num(r, "s_end_m") for r in seg50])
rseg50 = np.array([num(r, "resistance_Pa_s_m3") for r in seg50])
cum50 = np.r_[0, np.cumsum(rseg50)] / np.sum(rseg50)
x50 = np.r_[0, s50] * 1e3

fig, ax = plt.subplots(figsize=(8.3, 4.4))
ax.plot(slice_x, np.array([p_in,p_mid,p_out])/MMHG_PA, "o", ms=7, color="#222222", label="CFD sampled planes")
for row in rom:
    name = row["model"]
    if name == "CFD (reference)": continue
    model_dp = num(row, "dP_Pa")
    if name == "0D N=50":
        ax.plot(x50, (p_in-model_dp*cum50)/MMHG_PA, color=COLORS[name], lw=1.7, label=name)
    else:
        ax.plot([0, s_mm[-1]], [p_in/MMHG_PA, (p_in-model_dp)/MMHG_PA],
                color=COLORS.get(name), lw=1.5, label=name)
ax.set_xlabel("Arc length [mm]"); ax.set_ylabel("Pressure [mmHg]")
ax.set_title("Steady pressure reconstruction"); ax.legend(frameon=False, ncol=2)
savefig(fig, "07_pressure_model_comparison.png"); plt.show()


In [ ]:
complexity = [
    ("3D CFD", "≈10⁵–10⁷ cells", "3 velocity + pressure per cell", "High", "Full fields"),
    ("1D / hybrid", f"{len(profile)} axial elements", "Area/flow/pressure", "Low–moderate", "Axial waves/profiles"),
    ("0D native", "123 resistors", "Nodal pressure/flow", "Very low", "Network bulk values"),
    ("0D segmented", "50 resistors", "Nodal pressure/flow", "Very low", "Coarser axial bulk values"),
    ("Poiseuille", "1 element", "Δp and Q", "Negligible", "Global bulk relation"),
]
show_table(["Representation", "Spatial scale", "State", "Relative cost", "Recoverable output"], complexity)


### Why agreement improves as geometric information is retained

The one-element Poiseuille model has the largest error because it replaces the measured tortuous, variable-radius vessel with one straight circular tube and one representative radius. For this geometry that averaging is consequential: the radius varies substantially along the centerline, and resistance depends approximately on \(r^{-4}\). A mean radius therefore cannot reproduce the disproportionate pressure loss contributed by narrow sections.

The native 0D model improves the prediction because it retains the local radius and length of all 123 centerline elements. It still omits curvature-induced secondary flow and cross-sectional profile distortion, but it preserves where high-resistance regions occur. The 50-element model retains much of this distribution while averaging within larger elements. Segmentation generally reduces the geometric averaging error relative to a single tube; here, however, the native discretisation is more accurate than the 50-element representation because coarsening smooths some narrow regions. Thus the results support convergence through **preserving measured local geometry**, not the generic claim that any increase in segment count must monotonically reduce the error.

The distributed 1D-style representation gives the closest steady agreement because it retains the full axial variation of area and accumulates pressure loss along the centerline rather than collapsing the vessel to one global relation. Its remaining difference from CFD is consistent with the physics absent from the reduced representation, including three-dimensional velocity-profile distortion and curvature-related losses. This is a steady resistance comparison, not evidence that transient wall mechanics have been calibrated.


### Final comparison of representable physical information

| Quantity | CFD | Poiseuille | Native 0D | Segmented 0D | 1D |
|---|---|---|---|---|---|
| Pressure field | ✓, three-dimensional | No; bulk relation only | Approximate nodal values | Approximate nodal values | Approximate axial field |
| Pressure drop | ✓ | ✓ | ✓ | ✓ | ✓ |
| Flow rate | ✓ | ✓ | ✓ | ✓ | ✓ |
| Hydraulic resistance | ✓ | ✓ | ✓ | ✓ | ✓ |
| Pressure profile | ✓ | No | Piecewise | Piecewise | Axial |
| Velocity profile | ✓ | No | No | No | Approximate through closure |
| Wall shear stress | ✓ if wall gradients are saved | Analytical estimate | No | No | Approximate through closure |
| Secondary flow | ✓ | No | No | No | No in the present model |
| Vortices / recirculation | ✓ | No | No | No | No |

The reduced-order models reproduce global haemodynamic quantities—particularly pressure drop, flow rate, and hydraulic resistance—because these are integrated conservation quantities. Their increasing axial resolution improves the representation of where pressure is lost, but it does not restore the discarded three-dimensional velocity field. Detailed structures such as secondary flow, vortices, recirculation, and wall-resolved shear therefore remain exclusive to CFD in the present comparison.


## 5. Information preservation during simplification


In [ ]:
preservation = [
    ("3D lumen geometry", "Full", "Collapsed", "Collapsed", "No", "Lost dimensional detail"),
    ("Centerline/connectivity", "Extractable", "Preserved", "Preserved", "Yes", "Direct geometry transfer"),
    ("Radius/area", "Full surface", "Axial profile", "Element average", "Partly", "Resolution-dependent"),
    ("Pressure", "3D field", "Axial/nodal", "Nodal", "Partly", "Cross-sectional variation lost"),
    ("Pressure drop", "Preserved", "Preserved", "Preserved", "Yes", "Bulk target"),
    ("Flow rate", "Preserved", "Preserved", "Preserved", "Yes", "Bulk conserved quantity"),
    ("Pressure profile", "3D", "Approximated", "Piecewise", "Partly", "Axial only"),
    ("Velocity profile", "Resolved", "Assumed/reconstructed", "Absent", "Potentially", "Needs closure assumption"),
    ("Secondary flow", "Resolved", "Absent in standard 1D", "Absent", "No", "Lost"),
    ("Wall shear stress", "Resolved if wall gradients saved", "Estimated", "Absent", "Potentially", "Requires assumed profile"),
    ("Vortices", "Resolved", "Absent", "Absent", "No", "Lost"),
    ("Recirculation", "Resolved", "Absent", "Absent", "No", "Lost"),
]
show_table(["Quantity", "CFD", "1D", "0D", "Recoverable?", "Assessment"], preservation)


Reduction preserves integrated conservation laws better than spatial mechanisms. Pressure drop and flow are natural calibration targets because every model exposes them. Radius and area remain available but are increasingly averaged. Velocity-profile skewness, secondary motion, vortices, and recirculation are discarded state—not merely numerical error—and cannot be recovered uniquely from a 0D solution. A profile or WSS may be reconstructed only after adding a closure assumption, so it is a new model prediction rather than preserved CFD information.


## 6. Calibration analysis for a hybrid 1D–0D model


The calibration problem separates naturally into two parameter classes.

**Geometry-controlled quantities**—centerline length, local radius and area, tortuosity, curvature, and connectivity—are fixed once the centerline and lumen geometry have been extracted. Their measurement uncertainty should be propagated through a sensitivity analysis, but they should not be treated as unconstrained fitting parameters. Radius in particular must not be freely tuned to force agreement: because resistance scales approximately as \(r^{-4}\), even a small adjustment could hide missing model physics while contradicting the measured anatomy.

**Physiological and closure quantities** cannot be identified from lumen geometry alone. Outlet resistance represents the pressure–flow effect of the downstream circulation that is absent from the truncated 3D domain. An effective viscous or resistance correction accounts for losses not represented by the reduced equations and can be calibrated against the present steady \(\Delta p/Q\) target.

Wall stiffness, wall thickness, vessel compliance, and Windkessel compliance govern storage, pulse-wave speed, waveform phase, and diastolic decay. The present CFD solution is steady and uses no time-resolved pressure, flow, or area waveform, so these parameters are not identifiable from the available evidence. Reliable estimation would require transient simulations or measurements with pulsatile pressure and flow targets. The ranking below therefore distinguishes parameters that affect the current steady comparison from those that become influential only in a pulsatile calibration.


In [ ]:
calibration = [
    (1, "Viscosity / resistance correction", "Fluid/model closure", "No", "Very high", "Δp/Q and axial pressure", "Calibrate effective kR; retain measured μ as prior"),
    (2, "Outlet resistance", "Physiological boundary", "No", "Very high", "Mean pressure and flow split", "Tune to terminal pressure/flow targets"),
    (3, "Windkessel proximal/distal resistance", "Physiological boundary", "No", "High (pulsatile)", "Mean and pulsatile pressure", "Needs transient targets"),
    (4, "Wall stiffness E", "Wall physiology", "No", "High (pulsatile)", "Pulse-wave velocity/phase", "Not identifiable from steady CFD"),
    (5, "Wall thickness h", "Wall physiology", "No", "High with E", "Compliance/wave speed", "Use imaging/literature prior"),
    (6, "Outlet compliance", "Physiological boundary", "No", "Medium–high (pulsatile)", "Diastolic decay/phase", "Needs waveform target"),
    (7, "Vessel compliance", "Derived from E,h,r", "No", "Medium–high (pulsatile)", "Area/pressure waveform", "Calibrate jointly, avoid non-identifiability"),
    (8, "Length", "Geometry", "Yes", "Medium", "Resistance and wave transit", "Fixed from centerline"),
    (9, "Radius/area", "Geometry", "Yes", "Very high", "Resistance (≈r⁻⁴)", "Fixed with uncertainty study; do not freely tune"),
    (10, "Connectivity", "Geometry", "Yes", "Structural", "Flow paths/splits", "Fixed from graph"),
]
show_table(["Rank", "Parameter", "Class", "From geometry?", "Expected influence",
            "Observable affected", "Recommended treatment"], calibration)


In [ ]:
# Steady resistance calibration implied by each uncalibrated reduced model.
cal_rows = []
for r in rom:
    if r["model"] == "CFD (reference)": continue
    kR = R_cfd / num(r, "R_Pa_s_m3")
    cal_rows.append((r["model"], kR, (kR-1)*100,
                     "Effective multiplier needed to match steady CFD R"))
show_table(["Model", "kR = R_CFD/R_model", "Correction [%]", "Interpretation"], cal_rows)


### Recommended staged calibration

1. **Freeze geometry** (length, connectivity, radius/area) and quantify radius uncertainty rather than tuning it invisibly.
2. **Match steady resistance** using the smallest defensible viscous/tortuosity correction. For the native 0D model the implied multiplier is about 1.10.
3. **Set terminal resistance and flow split** from boundary targets.
4. Only with transient data, calibrate **\(E h^{-1}\)** or pulse-wave velocity, then Windkessel compliance. Estimating \(E\), \(h\), and compliance independently from the same waveform is poorly identifiable.
5. Validate on observables not used in fitting: axial pressure shape, waveform phase/amplitude, and branch flow split.


## 7. Conclusions


1. **Direct geometry transfer.** The reduced models inherit centerline arc length, local radius and area, connectivity, tortuosity, and curvature from the 3D anatomy. These quantities define the model skeleton without haemodynamic calibration.

2. **Haemodynamics reproduced by simplification.** Pressure drop, flow rate, and hydraulic resistance are the best-preserved quantities because they are global conservation measures available in every representation. The distributed 1D-style result gives the closest resistance agreement in the present steady comparison, followed by the native locally resolved 0D model.

3. **Information lost.** Collapsing the lumen removes the three-dimensional pressure and velocity fields. Standard 0D and 1D representations cannot retain cross-sectional velocity-profile distortion, secondary flow, vortices, recirculation, or wall-resolved shear. These quantities cannot be recovered uniquely from bulk pressure and flow.

4. **Why local geometry matters.** The single-tube Poiseuille approximation performs worst because it averages a tortuous vessel with a strongly varying radius. Retaining local length and radius preserves the narrow, high-resistance regions implied by the approximate \(r^{-4}\) dependence. Increasing spatial resolution is useful insofar as it reduces this geometric averaging; it does not restore omitted three-dimensional physics.

5. **Parameters requiring calibration.** Measured radius, length, and connectivity should remain fixed subject to geometric uncertainty. Effective resistance corrections and downstream outlet resistance require calibration to reproduce steady patient-specific pressure and flow. Wall stiffness, wall thickness, compliance, and Windkessel parameters require pulsatile data and cannot be identified reliably from the present steady CFD solution.

Overall, reduced-order models preserve the global haemodynamic response far better than the local flow structures that produce it. Their value is therefore strongest for efficient pressure–flow prediction and network simulation, provided that measured local geometry is retained and non-geometric boundary and wall parameters are calibrated against appropriate data.


In [ ]:
# Machine-readable headline results for report reuse.
headline = [
    ("main_path_length_mm", geom["main path arc length"], "mm"),
    ("tortuosity", geom["tortuosity  T = arc / straight"], "-"),
    ("cfd_pressure_drop_pa", dp, "Pa"),
    ("cfd_pressure_drop_mmhg", dp/MMHG_PA, "mmHg"),
    ("cfd_flow_rate_ml_s", q_ref*1e6, "mL/s"),
    ("cfd_resistance_pa_s_m3", R_cfd, "Pa s/m3"),
    ("flow_imbalance_pct", mass_imbalance, "%"),
]
with open(OUT/"headline_results.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["quantity", "value", "unit"]); w.writerows(headline)
show_table(["Output", "Location"],
           [("Figures and headline results", str(OUT.relative_to(ROOT))),
            ("FirstBlood input template", str(inputs["FirstBlood template"].relative_to(ROOT)))])
